In [15]:
# --- Instalaciones ---
# Selenium: Para controlar el navegador
# webdriver-manager: Para gestionar los drivers automáticamente (¡Olvídate de descargar .exe manualmente!)
# !pip install selenium webdriver-manager pandas
!pip install  requests bs4

# --- Importaciones ---
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By

# Importaciones para esperas explícitas (Lo veremos más adelante)
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Clase 3: Conferencia Práctica - Scraping Dinámico (Selenium)

**Objetivo:** Aprender a extraer datos de sitios web dinámicos que requieren interacción (clicks, scroll) o que cargan datos mediante JavaScript.

**El Problema:**
Hasta ahora, hemos usado `requests` para pedir el HTML al servidor. Pero, ¿qué pasa cuando el servidor nos envía un HTML "vacío" y es el navegador quien lo rellena después usando JavaScript?

**La Solución:**
Necesitamos una herramienta que simule ser un humano navegando: **Selenium**.

**Herramientas de hoy:**
1.  **Selenium:** Automatización de navegadores web.
2.  **Webdriver Manager:** Gestor automático de controladores de navegador.
3.  **Infinite Scroll:** Técnicas para manejar páginas que cargan más contenido al bajar.

## Parte 1: Diagnóstico - ¿Estático o Dinámico?

Antes de sacar la artillería pesada (Selenium), siempre debemos verificar si `requests` es suficiente.

Intentemos scrapear `http://quotes.toscrape.com/scroll` con el método de la clase anterior.

In [16]:
import requests
from bs4 import BeautifulSoup

url = "http://quotes.toscrape.com/scroll"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

# Intentamos buscar las citas (sabemos que están en divs con clase 'quote')
citas = soup.find_all('div', class_='quote')

print(f"Intentando scrapear con requests...")
print(f"Citas encontradas: {len(citas)}")

# Explicación:
# Si el resultado es 0, es porque las citas NO están en el HTML original.
# Se cargan dinámicamente vía JavaScript (JSON) cuando el navegador renderiza la página.

Intentando scrapear con requests...
Citas encontradas: 0


## Parte 2: Hola, Selenium

Como `requests` falló, usaremos Selenium. Selenium abre un navegador real (Chrome, Firefox, etc.), espera a que el JavaScript se ejecute, y nos permite interactuar con la página.

### Configuración del Driver
Para controlar Chrome, necesitamos un "chofer" (Driver). Usaremos `webdriver_manager` para que lo descargue e instale automáticamente según tu versión de Chrome.

In [17]:
# 1. Configuración del Servicio (El Manager descarga el driver correcto).
# Intentamos que webdriver_manager gestione el driver automáticamente;
# si falla, usamos el chromedriver incluido en el repo como fallback.
# try:
#     service = Service(ChromeDriverManager().install())
#     print("Usando chromedriver descargado por webdriver_manager.")
# except Exception as e:
#     # Fallback a driver local (asegúrate de tener permisos de ejecución)
# print("webdriver_manager falló: ", e)

service = Service('chromedriver-linux64/chromedriver')
print("Usando chromedriver local: chromedriver-linux64/chromedriver")
# 2. Opciones del Navegador
options = webdriver.ChromeOptions()
# Sugerencias útiles para ejecutar en Linux en entornos headless/servidor:
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
# options.add_argument('--headless') # Descomenta para modo headless

# 3. Iniciar el navegador
driver = webdriver.Chrome(service=service, options=options)

# 4. Navegar a la página
print("Abriendo navegador...")
driver.get("http://quotes.toscrape.com/scroll")

# Esperamos unos segundos para ver la página cargar
time.sleep(3)

print("¡Página cargada!")

Usando chromedriver local: chromedriver-linux64/chromedriver
Abriendo navegador...
Abriendo navegador...
¡Página cargada!
¡Página cargada!


### Selectores en Selenium

A diferencia de BeautifulSoup (`find`, `select`), Selenium tiene sus propios métodos para encontrar elementos, importados a través de `By`.

* `By.CLASS_NAME`: Similar a buscar por clase.
* `By.CSS_SELECTOR`: Igual que `soup.select()`.
* `By.XPATH`: Una forma muy potente de buscar elementos por su estructura XML (muy útil en scraping complejo).
* `By.ID`: Busca por ID único.

In [9]:
# Ejemplo: Encontrar elementos visibles inicialmente
citas_visibles = driver.find_elements(By.CLASS_NAME, "quote")
print(f"Citas visibles inicialmente: {len(citas_visibles)}")

# Extraer texto de la primera cita
if citas_visibles:
    primer_texto = citas_visibles[0].find_element(By.CLASS_NAME, "text").text
    print(f"Primera cita: {primer_texto}")

Citas visibles inicialmente: 10
Primera cita: “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”


## Parte 3: El Desafío del "Infinite Scroll"

La página `quotes.toscrape.com/scroll` tiene una particularidad: carga más citas a medida que bajas (scroll). Si solo abrimos la página, solo veremos las primeras 10.

**Estrategia para Scrollear:**
1.  Usar JavaScript a través de Selenium para bajar hasta el final de la página.
2.  Esperar a que cargue el nuevo contenido.
3.  Repetir hasta que no se carguen más elementos o alcancemos un límite.

In [18]:
# --- Configuración del Límite ---
MAX_SCROLLS = 3       # Queremos bajar solo 3 veces
scroll_count = 0      # Contador inicial
SCROLL_PAUSE_TIME = 2 # Tiempo de espera

# Altura inicial de la página
last_height = driver.execute_script("return document.body.scrollHeight")

# El bucle se ejecuta MIENTRAS el contador sea menor que el límite
while scroll_count < MAX_SCROLLS:
    print(f"Bajando... (Intento {scroll_count + 1} de {MAX_SCROLLS})")
    
    # 1. Bajar hasta el fondo
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

    # 2. Esperar a que cargue
    time.sleep(SCROLL_PAUSE_TIME)

    # 3. Calcular nueva altura
    new_height = driver.execute_script("return document.body.scrollHeight")

    # 4. Verificar si llegamos al final real de la página (por si hay menos datos de los que pedimos)
    if new_height == last_height:
        print("¡Llegamos al final de la página antes de completar los 3 scrolls!")
        break
    
    last_height = new_height
    
    # 5. ¡IMPORTANTE! Aumentar el contador
    scroll_count += 1

print("Proceso de scroll finalizado.")

Bajando... (Intento 1 de 3)
Bajando... (Intento 2 de 3)
Bajando... (Intento 2 de 3)
Bajando... (Intento 3 de 3)
Bajando... (Intento 3 de 3)
Proceso de scroll finalizado.
Proceso de scroll finalizado.


## Parte 4: Extracción y Guardado de Datos

Ahora que hemos desplegado todo el contenido de la página (el DOM está completo), podemos proceder a extraer la información tal como lo haríamos normalmente.

**Nota:** Podríamos pasar el HTML actual a BeautifulSoup (`driver.page_source`) si preferimos su sintaxis, pero lo haremos 100% con Selenium para practicar.

In [19]:
# 1. Encontrar todas las tarjetas de citas (ahora deberían ser muchas más)
citas_elementos = driver.find_elements(By.CLASS_NAME, "quote")
print(f"Total de citas encontradas tras el scroll: {len(citas_elementos)}")

data_citas = []

for cita in citas_elementos:
    try:
        # Extracción de texto
        texto = cita.find_element(By.CLASS_NAME, "text").text
        
        # Extracción de autor
        autor = cita.find_element(By.CLASS_NAME, "author").text
        
        # Extracción de etiquetas (puede haber varias)
        tags_elementos = cita.find_elements(By.CLASS_NAME, "tag")
        tags = [tag.text for tag in tags_elementos]
        
        data_citas.append({
            "texto": texto,
            "autor": autor,
            "tags": ", ".join(tags) # Unimos las tags en un string
        })
    except Exception as e:
        print(f"Error extrayendo una cita: {e}")

# Cerramos el navegador (¡Buena práctica!)
# driver.quit()

# Crear DataFrame
df = pd.DataFrame(data_citas)
print(df.head())

# Guardar CSV
df.to_csv("citas_scroll_selenium.csv", index=False, encoding='utf-8')
print("Archivo guardado exitosamente.")

Total de citas encontradas tras el scroll: 40
                                               texto            autor  \
0  “The world as we have created it is a process ...  Albert Einstein   
1  “It is our choices, Harry, that show what we t...     J.K. Rowling   
2  “There are only two ways to live your life. On...  Albert Einstein   
3  “The person, be it gentleman or lady, who has ...      Jane Austen   
4  “Imperfection is beauty, madness is genius and...   Marilyn Monroe   

                                           tags  
0        change, deep-thoughts, thinking, world  
1                            abilities, choices  
2  inspirational, life, live, miracle, miracles  
3              aliteracy, books, classic, humor  
4                    be-yourself, inspirational  
Archivo guardado exitosamente.
                                               texto            autor  \
0  “The world as we have created it is a process ...  Albert Einstein   
1  “It is our choices, Harry, that sho

## Parte 5: Interacción con Formularios y Logins

El scraping moderno no es solo "ver" datos, a veces necesitamos "entrar" a sitios protegidos. Selenium brilla aquí porque puede escribir y hacer clic como un humano.



**Nuevos Métodos:**
1.  `.click()`: Simula un clic izquierdo en un elemento (enlaces, botones).
2.  `.send_keys("texto")`: Simula la escritura del teclado en un campo de texto (`input`).

**El Reto:**
Vamos a iniciar sesión en el sitio.
1.  En la página principal, buscaremos el enlace que dice **Login**.
2.  Llenaremos el formulario (Usuario y Contraseña).
3.  Enviaremos el formulario.

In [20]:
# 1. Volvemos a la página inicial para empezar el flujo desde cero
driver.get("http://quotes.toscrape.com/scroll")
time.sleep(2)

# 2. Buscamos el enlace de Login
# En el HTML que vimos, es un <a> con texto "Login".
# Selenium tiene un selector especial para esto: By.LINK_TEXT
try:
    login_link = driver.find_element(By.LINK_TEXT, "Login")
    print("Enlace de Login encontrado.")
    
    # 3. Hacemos clic
    login_link.click()
    print("Navegando a la página de Login...")
    
    # Esperamos a que cargue la nueva página
    time.sleep(2)
    
    # Verificamos que estamos en la URL correcta
    print(f"URL actual: {driver.current_url}")

except Exception as e:
    print(f"No pudimos encontrar o hacer clic en el login: {e}")

Enlace de Login encontrado.
Navegando a la página de Login...
Navegando a la página de Login...
URL actual: http://quotes.toscrape.com/login
URL actual: http://quotes.toscrape.com/login


### Llenando los Inputs

Ahora estamos en `/login`. Si inspeccionamos esa página (puedes hacerlo en tu navegador real), veremos que:
* El input de usuario tiene `id="username"`.
* El input de contraseña tiene `id="password"`.
* El botón de ingresar es un `<input type="submit">`.

*Nota: En este sitio de prueba, cualquier usuario y contraseña son válidos.*

In [21]:
# Definimos credenciales falsas (para propósitos educativos)
USUARIO = "alumno_python"
PASSWORD = "password123"

try:
    # 1. Encontrar los elementos del formulario por su ID
    input_user = driver.find_element(By.ID, "username")
    input_pass = driver.find_element(By.ID, "password")
    
    # A veces el botón no tiene ID, podemos usar CSS Selector buscando el botón de submit
    boton_submit = driver.find_element(By.CSS_SELECTOR, "input[type='submit']") 

    # 2. Interactuar con los elementos
    print(f"Escribiendo usuario: {USUARIO}")
    input_user.send_keys(USUARIO) # Escribimos el usuario
    
    print("Escribiendo contraseña...")
    input_pass.send_keys(PASSWORD) # Escribimos la contraseña
    
    # 3. Enviar el formulario
    print("Haciendo clic en Login...")
    boton_submit.click()
    
    # Esperamos respuesta
    time.sleep(2)

    # 4. Verificación: ¿Cómo sabemos si funcionó?
    # Generalmente, aparece un botón de "Logout" o el texto "Welcome"
    try:
        logout_link = driver.find_element(By.LINK_TEXT, "Logout")
        print("¡LOGIN EXITOSO! Se encontró el botón de Logout.")
    except:
        print("Parece que el login falló, no veo el botón de Logout.")

except Exception as e:
    print(f"Error durante el proceso de login: {e}")

# Cerramos al finalizar la clase
# driver.quit()

Escribiendo usuario: alumno_python
Escribiendo contraseña...
Haciendo clic en Login...
Haciendo clic en Login...
¡LOGIN EXITOSO! Se encontró el botón de Logout.
¡LOGIN EXITOSO! Se encontró el botón de Logout.
